# 02 · Silver
Typed, cleaned and validated tables. Grain and cleaning decisions are stored as table comments.
Silver cleans but never filters: every table must keep its bronze row count.

In [0]:
from pyspark.sql import functions as F

from easymoney_pipeline.quality import (
    check_binary, check_in_domain, check_no_orphans, check_no_parse_loss,
    check_not_null, check_row_count, check_unique,
)
from easymoney_pipeline.transforms import split_segment, to_binary_flag

BRONZE = "workspace.easymoney_bronze"
SILVER = "workspace.easymoney_silver"
KEYS = ["customer_id", "partition_month"]


def write_silver(df, table: str, comment: str) -> None:
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", True)
       .saveAsTable(f"{SILVER}.{table}"))
    spark.sql(f"COMMENT ON TABLE {SILVER}.{table} IS '{comment}'")
    n = check_row_count(spark.table(f"{SILVER}.{table}"),
                        spark.table(f"{BRONZE}.{table}").count(), f"silver.{table}")
    print(f"{table:<30} {n:>12,} rows")

## customer_products

In [0]:
PRODUCT_COLS = [
    "short_term_deposit", "loans", "mortgage", "funds", "securities",
    "long_term_deposit", "em_account_pp", "credit_card", "payroll",
    "pension_plan", "payroll_account", "emc_account", "debit_card",
    "em_account_p", "em_acount",
]
RENAMES = {"em_acount": "em_account"}   # typo in source
FLAG_COLS = [RENAMES.get(c, c) for c in PRODUCT_COLS]

products = spark.table(f"{BRONZE}.customer_products").select(
    F.col("pk_cid").cast("int").alias("customer_id"),
    F.to_date("pk_partition", "yyyy-MM").alias("partition_month"),
    # 61 NULLs in payroll / pension_plan (0.001%) -> 0
    *[to_binary_flag(c).alias(RENAMES.get(c, c)) for c in PRODUCT_COLS],
)

In [0]:
check_unique(products, KEYS)
check_binary(products, FLAG_COLS)

In [0]:
write_silver(products, "customer_products",
    "Product holdings per customer and month. Grain: customer_id + partition_month. "
    "61 NULLs in payroll and pension_plan set to 0 (0.001 pct of rows). "
    "em_acount renamed to em_account.")

## customer_sociodemographics

In [0]:
socio = spark.table(f"{BRONZE}.customer_sociodemographics").select(
    F.col("pk_cid").cast("int").alias("customer_id"),
    F.to_date("pk_partition", "yyyy-MM").alias("partition_month"),
    "country_id",
    F.col("region_code").cast("double").cast("int").alias("region_code"),   # "29.0" -> 29, ES only
    "gender",                                                               # H/V undefined in dictionary: kept as is
    F.col("age").cast("int").alias("age"),
    F.when(F.col("deceased") == "S", True)
     .when(F.col("deceased") == "N", False).alias("is_deceased"),
    F.round(F.col("salary").cast("double"), 2).alias("salary"),             # ~26% NULL, imputed at training time
)

In [0]:
check_unique(socio, KEYS)
check_in_domain(socio, "is_deceased", [True, False])   # NULL = unexpected source code
check_in_domain(socio, "gender", ["H", "V"], allow_null=True)

In [0]:
write_silver(socio, "customer_sociodemographics",
    "Sociodemographics per customer and month. Grain: customer_id + partition_month. "
    "salary ~26 pct NULL, not imputed here (imputation fitted on train months only). "
    "region_code applies to ES only. gender codes H/V kept as in source.")

## customer_commercial_activity

In [0]:
segment_code, segment_name = split_segment("segment")

activity_staged = spark.table(f"{BRONZE}.customer_commercial_activity").select(
    F.col("pk_cid").cast("int").alias("customer_id"),
    F.to_date("pk_partition", "yyyy-MM").alias("partition_month"),
    F.col("entry_date").alias("_raw_entry_date"),
    F.col("segment").alias("_raw_segment"),
    F.expr("CAST(try_to_timestamp(entry_date, 'yyyy-MM') AS DATE)").alias("entry_month"),
    "entry_channel",
    F.col("active_customer").cast("double").cast("int").alias("is_active"),   # "1.0" -> 1
    segment_code.alias("segment_code"),                                       # "03 - UNIVERSITARIO" -> 3
    segment_name.alias("segment_name"),                                       # -> "UNIVERSITARIO"
)

In [0]:
check_unique(activity_staged, KEYS)
check_in_domain(activity_staged, "is_active", [0, 1])
# try_* parsers return NULL instead of failing: make sure nothing was lost
check_no_parse_loss(activity_staged, "_raw_entry_date", "entry_month")
check_no_parse_loss(activity_staged, "_raw_segment", "segment_code")

activity = activity_staged.drop("_raw_entry_date", "_raw_segment")

In [0]:
write_silver(activity, "customer_commercial_activity",
    "Commercial activity per customer and month. Grain: customer_id + partition_month. "
    "entry_date (yyyy-MM) parsed to entry_month. segment split into segment_code and segment_name. "
    "NULL entry_channel and segment (~2 pct, mostly the same customers) kept; handled in gold.")

## product_description

In [0]:
catalog = spark.table(f"{BRONZE}.product_description").select(
    F.col("pk_product_id").cast("double").cast("int").alias("product_id"),
    F.when(F.trim("product_desc") == "em_acount", "em_account")   # align with customer_products
     .otherwise(F.trim("product_desc")).alias("product_name"),
    F.trim("family_product").alias("product_family"),
)

check_not_null(catalog, ["product_id"])
check_unique(catalog, ["product_id"])

In [0]:
write_silver(catalog, "product_description",
    "Product catalog. Grain: product_id. product_name em_acount renamed to em_account "
    "to match customer_products.")

## sales

In [0]:
# Accepts '2018-06-01' and '2018-06'; normalised to the first day of the month
MONTH_SALE = "coalesce(try_to_timestamp(month_sale, 'yyyy-MM-dd'), try_to_timestamp(month_sale, 'yyyy-MM'))"

sales_staged = spark.table(f"{BRONZE}.sales").select(
    F.col("pk_sale").cast("double").cast("long").alias("sale_id"),
    F.col("cid").cast("double").cast("int").alias("customer_id"),
    F.col("month_sale").alias("_raw_month_sale"),
    F.expr(f"CAST(date_trunc('MONTH', {MONTH_SALE}) AS DATE)").alias("sale_month"),
    F.col("product_id").cast("double").cast("int").alias("product_id"),
    F.round(F.col("net_margin").cast("double"), 2).alias("net_margin"),
)

In [0]:
check_unique(sales_staged, ["sale_id"])
check_not_null(sales_staged, ["sale_id", "customer_id", "product_id"])
check_no_parse_loss(sales_staged, "_raw_month_sale", "sale_month")
check_no_orphans(sales_staged, catalog, "product_id")   # every sale maps to a catalog product

sales = sales_staged.drop("_raw_month_sale")

In [0]:
write_silver(sales, "sales",
    "Product sales. Grain: sale_id. sale_month normalised to the first day of the month. "
    "Referential integrity with product_description validated.")